In [ ]:
# 1. Install dependencies
!pip install -q bitsandbytes peft python-chess


In [ ]:
# 2. Find adapter and load model
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "microsoft/Phi-3.5-mini-instruct"

adapter_dir = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "adapter_config.json" in files:
        adapter_dir = root
        break

print(f"Using adapter_dir: {adapter_dir}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading base model and tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
)

print(f"Attaching fine-tuned LoRA adapter from {adapter_dir}...")
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()
model.config.use_cache = False
print("Model ready for inference!")


In [ ]:
# 3. Validation functions: Scaffolding Leakage & Chess Grounding (with negative assertion support)
import re, json, chess
from dataclasses import dataclass
from typing import Optional, List, Tuple

@dataclass(frozen=True)
class FilterResult:
    passed: bool
    reason: str = "ok"

BANNED_SCAFFOLDING_PATTERNS = [
    re.compile(r"\bcritical\s+instruction\b", re.IGNORECASE),
    re.compile(r"\binstruction\s+(?:indicates|points|states|notes|requires|suggests)\b", re.IGNORECASE),
    re.compile(r"\bas\s+(?:instructed|provided\s+above|given\s+above|shown\s+above)\b", re.IGNORECASE),
    re.compile(r"\btactical\s+continuation\s+(?:shows|indicates|points|suggests)\b", re.IGNORECASE),
    re.compile(r"\b(?:the\s+)?verified\s+(?:tactical\s+)?continuation\b", re.IGNORECASE),
    re.compile(r"\bcontinuation\s+(?:provided|given|listed|above)\b", re.IGNORECASE),
    re.compile(r"\b(?:scaffolding|system\s+prompt)\b", re.IGNORECASE),
]

def validate_no_scaffolding_leakage(commentary: str) -> FilterResult:
    for pat in BANNED_SCAFFOLDING_PATTERNS:
        match = pat.search(commentary)
        if match:
            return FilterResult(passed=False, reason=f"Scaffolding leakage detected: '{match.group(0)}'")
    return FilterResult(passed=True)

PIECE_PATTERN = re.compile(r'\b(queen|rook|bishop|knight|pawn|king)\s+(?:on|at|to|from)\s+([a-h][1-8])\b', re.IGNORECASE)
NEGATIVE_ASSERTION_PREFIX = re.compile(r'\b(?:absence\s+of|lack\s+of|without\s+a|without\s+any|without|no|neither)\s+(?:a\s+|an\s+|the\s+|any\s+)?$', re.IGNORECASE)
PIECE_MAP = {
    'queen': chess.QUEEN, 'rook': chess.ROOK, 'bishop': chess.BISHOP,
    'knight': chess.KNIGHT, 'pawn': chess.PAWN, 'king': chess.KING,
}
UNICODE_HYPHENS = re.compile(r'[\u2010\u2011\u2012\u2013\u2014\u2015\u2212]')
HYPHEN_MOVE_PATTERN = re.compile(r'(?<!\w)(?:(?P<prefix>\d+\.+|\.{2,3})\s*)?(?P<san>[NBRQK]?[a-h][1-8][-–][a-h][1-8][+#]?)(?!\w)', re.UNICODE)
STANDARD_SAN_PATTERN = re.compile(r'(?<!\w)(?:(?P<prefix>\d+\.+|\.{2,3})\s*)?(?P<san>O-O(?:-O)?[+#]?|[NBRQK][a-h1-8]?x?[a-h][1-8](?:=[NBRQK])?[+#]?|[a-h]x[a-h][1-8](?:=[NBRQK])?[+#]?|[a-h][1-8]=[NBRQK][+#]?|[a-h][1-8][+#])(?!\w)', re.UNICODE)
EXPLICIT_PAWN_PATTERN = re.compile(r'(?:\b(?:move|plays|played)\s+|(?:\d+\.+|\.{2,3})\s*)([a-h][1-8])\b', re.IGNORECASE)

def extract_piece_square_mentions(commentary: str) -> List[Tuple[str, str, bool]]:
    mentions = []
    for m in PIECE_PATTERN.finditer(commentary):
        piece_name = m.group(1).lower()
        sq_name = m.group(2).lower()
        preceding = commentary[max(0, m.start() - 35):m.start()]
        is_neg = bool(NEGATIVE_ASSERTION_PREFIX.search(preceding))
        mentions.append((piece_name, sq_name, is_neg))
    return mentions

def extract_algebraic_move_references(commentary: str) -> List[Tuple[str, str]]:
    commentary = UNICODE_HYPHENS.sub('-', commentary)
    moves = []
    for m in HYPHEN_MOVE_PATTERN.finditer(commentary):
        moves.append((m.group('san'), m.group(0), m.start()))
    for m in STANDARD_SAN_PATTERN.finditer(commentary):
        moves.append((m.group('san'), m.group(0), m.start()))
    for m in EXPLICIT_PAWN_PATTERN.finditer(commentary):
        moves.append((m.group(1), m.group(0), m.start()))
    unique_moves = []
    seen_spans = []
    for san, raw, start in sorted(moves, key=lambda x: x[2]):
        end = start + len(raw)
        if any(not (end <= s or start >= e) for s, e in seen_spans):
            continue
        seen_spans.append((start, end))
        unique_moves.append((san.strip().lstrip('.').rstrip('!?'), raw.strip()))
    return unique_moves

def is_legal_notation_on_board(b: chess.Board, notation: str) -> bool:
    hyphen_m = re.match(r'^([NBRQK]?)([a-h][1-8])[-–]([a-h][1-8])([+#]?)$', notation)
    if hyphen_m:
        from_sq = chess.parse_square(hyphen_m.group(2))
        to_sq = chess.parse_square(hyphen_m.group(3))
        candidate_move = chess.Move(from_sq, to_sq)
        if candidate_move in b.legal_moves:
            return True
        for promo in [chess.QUEEN, chess.ROOK, chess.BISHOP, chess.KNIGHT]:
            if chess.Move(from_sq, to_sq, promotion=promo) in b.legal_moves:
                return True
        return False
    try:
        m = b.parse_san(notation)
        return m in b.legal_moves
    except Exception:
        pass
    try:
        m = chess.Move.from_uci(notation)
        return m in b.legal_moves
    except Exception:
        pass
    return False

def is_destination_for_piece(b: chess.Board, expected_type: chess.PieceType, target_sq: chess.Square) -> bool:
    for m in b.legal_moves:
        if m.to_square == target_sq:
            piece = b.piece_at(m.from_square)
            if piece and piece.piece_type == expected_type:
                return True
    return False

def is_piece_square_grounded_in_tree(board_before, played_move, best_move, expected_type, target_sq):
    p = board_before.piece_at(target_sq)
    if p and p.piece_type == expected_type:
        return True
    if is_destination_for_piece(board_before, expected_type, target_sq):
        return True
    b_opp = board_before.copy()
    b_opp.turn = not board_before.turn
    if is_destination_for_piece(b_opp, expected_type, target_sq):
        return True
    if played_move and played_move in board_before.legal_moves:
        board_after = board_before.copy()
        board_after.push(played_move)
        p2 = board_after.piece_at(target_sq)
        if p2 and p2.piece_type == expected_type:
            return True
        if is_destination_for_piece(board_after, expected_type, target_sq):
            return True
        for opp_m in list(board_after.legal_moves)[:30]:
            b_ply3 = board_after.copy()
            b_ply3.push(opp_m)
            p3 = b_ply3.piece_at(target_sq)
            if p3 and p3.piece_type == expected_type:
                return True
            if is_destination_for_piece(b_ply3, expected_type, target_sq):
                return True
    if best_move and best_move in board_before.legal_moves:
        board_best = board_before.copy()
        board_best.push(best_move)
        p_best = board_best.piece_at(target_sq)
        if p_best and p_best.piece_type == expected_type:
            return True
        if is_destination_for_piece(board_best, expected_type, target_sq):
            return True
        for opp_m in list(board_best.legal_moves)[:30]:
            b_best_ply3 = board_best.copy()
            b_best_ply3.push(opp_m)
            p3 = b_best_ply3.piece_at(target_sq)
            if p3 and p3.piece_type == expected_type:
                return True
            if is_destination_for_piece(b_best_ply3, expected_type, target_sq):
                return True
    return False

def is_move_legal_in_position_tree(board_before, played_move, best_move, notation):
    if is_legal_notation_on_board(board_before, notation):
        return True
    b_opp = board_before.copy()
    b_opp.turn = not board_before.turn
    if is_legal_notation_on_board(b_opp, notation):
        return True
    if played_move and played_move in board_before.legal_moves:
        board_after = board_before.copy()
        board_after.push(played_move)
        if is_legal_notation_on_board(board_after, notation):
            return True
        for opp_m in list(board_after.legal_moves)[:30]:
            b_ply3 = board_after.copy()
            b_ply3.push(opp_m)
            if is_legal_notation_on_board(b_ply3, notation):
                return True
    if best_move and best_move in board_before.legal_moves:
        board_best = board_before.copy()
        board_best.push(best_move)
        if is_legal_notation_on_board(board_best, notation):
            return True
        for opp_m in list(board_best.legal_moves)[:30]:
            b_best_ply3 = board_best.copy()
            b_best_ply3.push(opp_m)
            if is_legal_notation_on_board(b_best_ply3, notation):
                return True
    return False

def validate_chess_grounding(fen: str, move_uci: str, commentary: str, best_move_uci: Optional[str] = None) -> FilterResult:
    try:
        board_before = chess.Board(fen)
        move = chess.Move.from_uci(move_uci) if move_uci else None
        best_move = chess.Move.from_uci(best_move_uci) if best_move_uci else None
    except Exception as e:
        return FilterResult(passed=False, reason=f"Invalid chess state: {e}")

    mentions = extract_piece_square_mentions(commentary)
    for piece_name, sq_name, is_negative in mentions:
        target_sq = chess.parse_square(sq_name)
        expected_type = PIECE_MAP.get(piece_name)
        if expected_type is None:
            continue
        if is_negative:
            p_current = board_before.piece_at(target_sq)
            if p_current and p_current.piece_type == expected_type:
                return FilterResult(passed=False, reason=f"Hallucination detected: Claimed absence of {piece_name} on {sq_name}, but {piece_name} is present")
        else:
            if not is_piece_square_grounded_in_tree(board_before, move, best_move, expected_type, target_sq):
                return FilterResult(passed=False, reason=f"Hallucination detected: No {piece_name} on or moving to square {sq_name}")

    move_refs = extract_algebraic_move_references(commentary)
    for clean_san, raw_token in move_refs:
        if not is_move_legal_in_position_tree(board_before, move, best_move, clean_san):
            return FilterResult(passed=False, reason=f"Illegal move mentioned in commentary: '{raw_token}' is not legal in this position tree")

    return FilterResult(passed=True)
print("Validation checkers loaded successfully!")


In [ ]:
# 4. Multi-Configuration Generation & Joint Evaluation
smoke_positions = [
  {
    "category": "none",
    "fen": "5r2/p7/2k5/8/3R4/P1P1pp1P/1P6/5K2 b - - 3 40",
    "move_san": "f2",
    "move_uci": "f3f2",
    "color": "Black",
    "engine": "GOOD (loss: 0 cp)",
    "motif": "none",
    "best_move": "f3f2",
    "generated_commentary": "The move f2-f3 by Black aims to disrupt White's pawn structure and open lines for potential penetration into the enemy camp. However, it's a slow and passive move that doesn't immediately address the central tension or create any immediate threats. White's pieces are more actively placed and could potentially exploit Black's lack of development. Strategically, Black needs to focus on piece development and king safety while considering White's central control. This move doesn't significantly contribute to Black's plan or pose an immediate challenge to White's position.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: 5r2/p7/2k5/8/3R4/P1P1pp1P/1P6/5K2 b - - 3 40\nMove: f2 (f3f2) by Black\nEngine: GOOD (loss: 0 cp)\nTactical Motif: none<|end|>\n<|assistant|>\n"
  },
  {
    "category": "fork",
    "fen": "4Q3/p3r3/1pp1k1pR/5p2/4p1q1/8/PP3P2/5K2 w - - 4 47",
    "move_san": "Rxg6+",
    "move_uci": "h6g6",
    "color": "White",
    "engine": "GOOD (loss: 0 cp)",
    "motif": "fork",
    "best_move": "h6g6",
    "generated_commentary": "White's move Rxg6+ is a tactical decision aimed at exploiting the fork motif. By removing the black rook on g8, White creates a double threat: the black king on h6 is attacked by the rook on g6, and the black pawn on h6 is attacked by the pawn on g6. This forces Black to respond, likely leading to a loss of material or a checkmate. The move is strong tactically and demonstrates White'norchess understanding of creating multiple threats to overwhelm the opponent's defenses. The engine evaluation of GOOD suggests that White's position is winning or at least significantly better after this move.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: 4Q3/p3r3/1pp1k1pR/5p2/4p1q1/8/PP3P2/5K2 w - - 4 47\nMove: Rxg6+ (h6g6) by White\nEngine: GOOD (loss: 0 cp)\nTactical Motif: fork<|end|>\n<|assistant|>\n"
  },
  {
    "category": "hanging_piece",
    "fen": "3rr1k1/pb2q1b1/1p2p2p/3nN1p1/1P2Q3/1B5P/P2B1PP1/1R2R1K1 b - - 0 25",
    "move_san": "Bxe5",
    "move_uci": "g7e5",
    "color": "Black",
    "engine": "MISTAKE (loss: 283 cp, best move was d5f4)",
    "motif": "hanging_piece",
    "best_move": "d5f4",
    "generated_commentary": "Black's move g7xe5 is a tactical mistake because it allows White to exploit the hanging piece after Bxe5. Instead of capturing the bishop, Black should have played d5xf4, attacking the bishop and gaining a tempo by attacking the rook on a1. This would have created pressure on White'northerner rook and potentially led to a better position or material advantage. The move g7xe5 loses Black's queen's rook without sufficient compensation, making it a poor strategic choice.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: 3rr1k1/pb2q1b1/1p2p2p/3nN1p1/1P2Q3/1B5P/P2B1PP1/1R2R1K1 b - - 0 25\nMove: Bxe5 (g7e5) by Black\nEngine: MISTAKE (loss: 283 cp, best move was d5f4)\nTactical Motif: hanging_piece\n\nTactical Continuation:\n- Engine Best Move: Nf4\n- Legal Opponent Response (ply+1): Qc2\nBase your explanation only on the position, the move played, and the line shown above \u2014 don't introduce additional moves, checks, or captures beyond what's given here or already visible on the board.<|end|>\n<|assistant|>\n"
  },
  {
    "category": "zwischenzug",
    "fen": "2rqr1k1/1b3pbp/pp1p2p1/n3p3/P2P1nP1/2P2N1P/BPQN1P1B/R3R1K1 b - - 4 18",
    "move_san": "Ne6",
    "move_uci": "f4e6",
    "color": "Black",
    "engine": "BLUNDER (loss: 344 cp, best move was f4h3)",
    "motif": "zwischenzug",
    "best_move": "f4h3",
    "generated_commentary": "The move Ne6 by Black is a zwischenzug, attempting to break the pin on the d6 knight by attacking the e4 pawn. However, it's flawed as it doesn'eed immediately resolve the pinning issue and instead allows White to play f4xe6, opening lines towards Black's king and doubling the pawns after exf4. A better strategic move would be f4h3, targeting the bishop on c8 and preparing to open the center with a pawn break.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: 2rqr1k1/1b3pbp/pp1p2p1/n3p3/P2P1nP1/2P2N1P/BPQN1P1B/R3R1K1 b - - 4 18\nMove: Ne6 (f4e6) by Black\nEngine: BLUNDER (loss: 344 cp, best move was f4h3)\nTactical Motif: zwischenzug\n\nTactical Continuation:\n- Engine Best Move: Nxh3+\n- Legal Opponent Response (ply+1): Kf1\nBase your explanation only on the position, the move played, and the line shown above \u2014 don't introduce additional moves, checks, or captures beyond what's given here or already visible on the board.<|end|>\n<|assistant|>\n"
  },
  {
    "category": "missed_sacrifice",
    "fen": "8/8/2Q2bk1/2Bp2p1/1P1Pp3/1NK1PpP1/q7/8 w - - 0 51",
    "move_san": "Qxd5",
    "move_uci": "c6d5",
    "color": "White",
    "engine": "MISTAKE (loss: 207 cp, best move was c5e7)",
    "motif": "missed_sacrifice",
    "best_move": "c5e7",
    "generated_commentary": "White's move Qxd5 is a tactical oversight as it allows Black to play c6xf3, winning the queen after a discovered attack on the rook on f1. The missed sacrifice c5xe7 would have been stronger, leading to a winning position by attacking the bishop on b7 and creating double threats. White's move is flawed as it fails to recognize the tactical motif of a missed sacrifice to gain a decisive material advantage.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: 8/8/2Q2bk1/2Bp2p1/1P1Pp3/1NK1PpP1/q7/8 w - - 0 51\nMove: Qxd5 (c6d5) by White\nEngine: MISTAKE (loss: 207 cp, best move was c5e7)\nTactical Motif: missed_sacrifice\n\nTactical Continuation:\n- Engine Best Move: Be7\n- Legal Opponent Response (ply+1): Kh5\nBase your explanation only on the position, the move played, and the line shown above \u2014 don't introduce additional moves, checks, or captures beyond what's given here or already visible on the board.<|end|>\n<|assistant|>\n"
  },
  {
    "category": "trapped_queen",
    "fen": "3r1r1k/2p1q3/p6p/n2QPp2/1b1P4/2N2N1P/PP4P1/R4RK1 w - - 2 23",
    "move_san": "Rac1",
    "move_uci": "a1c1",
    "color": "White",
    "engine": "MISTAKE (loss: 232 cp, best move was a2a3)",
    "motif": "trapped_queen",
    "best_move": "a2a3",
    "generated_commentary": "White's move Rac1 was a tactical error as it allows Black to execute a discovered check with the queen, putting White'nmistake\" into play. The best move would have been a2-a3, which would have helped to safeguard the White king and prevent the discovered check, maintaining a solid position without immediately falling into tactical vulnerability. This move would have also kept pressure on Black's position, potentially leading to better strategic prospects.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: 3r1r1k/2p1q3/p6p/n2QPp2/1b1P4/2N2N1P/PP4P1/R4RK1 w - - 2 23\nMove: Rac1 (a1c1) by White\nEngine: MISTAKE (loss: 232 cp, best move was a2a3)\nTactical Motif: trapped_queen<|end|>\n<|assistant|>\n"
  },
  {
    "category": "missed_mate",
    "fen": "1r5k/5Qp1/5q1p/3B4/p1P3P1/7P/P4P2/4R1K1 w - - 3 40",
    "move_san": "Qxf6",
    "move_uci": "f7f6",
    "color": "White",
    "engine": "BLUNDER (loss: 2366 cp, best move was e1e8)",
    "motif": "missed_mate",
    "best_move": "e1e8",
    "generated_commentary": "White's move Qxf6 is a tactical blunder as it overlooks the discovered check from the black bishop on g7, which could have led to a quick mate in one. The move fails to address the strategic threat posed by the bishop and allows Black to consolidate their pieces without immediate danger. The best move would have been e1e8, which would have activated the queen and connected the rooks, improving White'settings and preparing for potential tactical opportunities. This move would have been far superior strategically and tactically.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: 1r5k/5Qp1/5q1p/3B4/p1P3P1/7P/P4P2/4R1K1 w - - 3 40\nMove: Qxf6 (f7f6) by White\nEngine: BLUNDER (loss: 2366 cp, best move was e1e8)\nTactical Motif: missed_mate\n\nTactical Continuation:\n- Engine Best Move: Re8+\n- Forced Mating Line (ply+1 to ply+2): Rxe8 Qxe8+\nBase your explanation only on the position, the move played, and the line shown above \u2014 don't introduce additional moves, checks, or captures beyond what's given here or already visible on the board.<|end|>\n<|assistant|>\n"
  },
  {
    "category": "checkmate",
    "fen": "r2qrk2/p2n2p1/b1pb1n1p/8/1p1P1N2/1Q3N1P/PP3PP1/R1B1R1K1 w - - 2 17",
    "move_san": "Ng6#",
    "move_uci": "f4g6",
    "color": "White",
    "engine": "GOOD (loss: 0 cp)",
    "motif": "checkmate",
    "best_move": "f4g6",
    "generated_commentary": "White delivers checkmate with Ng6#, exploiting the pin on the f7 pawn by the queen's knight and the lack of adeblemarked pawns or pieces to block or capture the knight. This move is strong as it ends the game instantly, showcasing White's tactical understanding of exploiting weaknesses and creating a decisive threat. The tactical motif here is a checkmate delivered by a knight, which is a classic and powerful finishing technique in such endgames. The move is perfectly justified given the positional and tactical circumstances.",
    "full_serving_prompt": "<|system|>\nYou are a grandmaster chess commentator. Provide concise, 2-4 sentence expert commentary explaining why the move was played, whether it is strong or flawed, and its tactical/strategic justification. State the moves and consequences directly as your own grandmaster calculation; never refer to prompts, instructions, or provided lines.<|end|>\n<|user|>\nPosition: r2qrk2/p2n2p1/b1pb1n1p/8/1p1P1N2/1Q3N1P/PP3PP1/R1B1R1K1 w - - 2 17\nMove: Ng6# (f4g6) by White\nEngine: GOOD (loss: 0 cp)\nTactical Motif: checkmate\n\nTactical Continuation:\n- Engine Best Move: Ng6#\n- Forced Mating Line (ply+1 to ply+2): None (delivers immediate checkmate)\nBase your explanation only on the position, the move played, and the line shown above \u2014 don't introduce additional moves, checks, or captures beyond what's given here or already visible on the board.<|end|>\n<|assistant|>\n"
  }
]

# Build configured eos_token_id list including <|end|> (32007)
eos_ids = [tokenizer.eos_token_id]
end_id = tokenizer.convert_tokens_to_ids('<|end|>')
if end_id is not None and end_id not in eos_ids:
    eos_ids.append(end_id)
print(f"Configured eos_token_ids: {eos_ids}")

# Grounding instruction appended to prompts without continuation
grounding_closure = "Base your explanation only on the position and the move played — don't introduce additional moves, checks, or captures beyond what's visible on the board."

configurations = [
    {'id': 'greedy_std_max85', 'name': 'Greedy Standard (85 tok)', 'prompt_mode': 'standard', 'max_new_tokens': 85, 'do_sample': False, 'temperature': 0.0, 'repetition_penalty': 1.0},
    {'id': 'greedy_std_rep1.1', 'name': 'Greedy RepPenalty 1.1 (85 tok)', 'prompt_mode': 'standard', 'max_new_tokens': 85, 'do_sample': False, 'temperature': 0.0, 'repetition_penalty': 1.1},
    {'id': 'greedy_unified_max85', 'name': 'Greedy Unified Constraint (85 tok)', 'prompt_mode': 'unified', 'max_new_tokens': 85, 'do_sample': False, 'temperature': 0.0, 'repetition_penalty': 1.0},
    {'id': 'greedy_unified_rep1.1', 'name': 'Greedy Unified Rep1.1 (85 tok)', 'prompt_mode': 'unified', 'max_new_tokens': 85, 'do_sample': False, 'temperature': 0.0, 'repetition_penalty': 1.1},
    {'id': 'sample_unified_t0.3', 'name': 'Sampled T0.3 Unified (85 tok)', 'prompt_mode': 'unified', 'max_new_tokens': 85, 'do_sample': True, 'temperature': 0.3, 'top_p': 0.9, 'repetition_penalty': 1.05},
    {'id': 'greedy_std_max120', 'name': 'Greedy Standard (120 tok)', 'prompt_mode': 'standard', 'max_new_tokens': 120, 'do_sample': False, 'temperature': 0.0, 'repetition_penalty': 1.0},
]

all_eval_summaries = []
best_results = None
best_score = -1
best_cfg_id = None

for cfg in configurations:
    cfg_name = cfg['name']
    print('=' * 80)
    print(f"EVALUATING CONFIGURATION: {cfg_name}")
    print('=' * 80)
    cfg_results = []
    scaffolding_leaks = 0
    grounding_passes = 0

    for idx, p in enumerate(smoke_positions, 1):
        prompt = p['full_serving_prompt']
        if cfg['prompt_mode'] == 'unified' and 'Tactical Continuation:' not in prompt:
            parts = prompt.split('<|end|>\n<|assistant|>')
            prompt = parts[0].rstrip() + '\n' + grounding_closure + '<|end|>\n<|assistant|>\n'

        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        gen_kwargs = {
            'max_new_tokens': cfg['max_new_tokens'],
            'do_sample': cfg['do_sample'],
            'pad_token_id': tokenizer.eos_token_id,
            'eos_token_id': eos_ids,
            'use_cache': False,
        }
        if cfg.get('repetition_penalty', 1.0) != 1.0:
            gen_kwargs['repetition_penalty'] = cfg['repetition_penalty']
        if cfg['do_sample']:
            gen_kwargs['temperature'] = cfg['temperature']
            if 'top_p' in cfg:
                gen_kwargs['top_p'] = cfg['top_p']

        with torch.no_grad():
            outputs = model.generate(**inputs, **gen_kwargs)

        gen_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        comm = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

        chk_leak = validate_no_scaffolding_leakage(comm)
        chk_ground = validate_chess_grounding(p['fen'], p['move_uci'], comm, p.get('best_move'))

        if not chk_leak.passed:
            scaffolding_leaks += 1
        if chk_ground.passed:
            grounding_passes += 1

        p_eval = dict(p)
        p_eval['generated_commentary'] = comm
        p_eval['scaffolding_leak'] = not chk_leak.passed
        p_eval['leakage_reason'] = chk_leak.reason
        p_eval['grounding_passed'] = chk_ground.passed
        p_eval['grounding_reason'] = chk_ground.reason
        cfg_results.append(p_eval)

        status_str = f"Leak: {'FAIL' if not chk_leak.passed else 'CLEAN'} | Ground: {'PASS' if chk_ground.passed else 'FAIL'}"
        print(f"[{p['category']}] {p['move_san']}: {status_str}")
        print(f"   Commentary: {comm}")

    summary = {
        'id': cfg['id'],
        'name': cfg_name,
        'scaffolding_leaks': scaffolding_leaks,
        'grounding_passes': grounding_passes,
        'pass_rate': f"{grounding_passes}/8 ({grounding_passes/8*100:.1f}%)",
    }
    all_eval_summaries.append(summary)
    print(f"Summary for {cfg_name}: Grounding {grounding_passes}/8, Leaks: {scaffolding_leaks}/8")

    score = (100 if scaffolding_leaks == 0 else 0) + grounding_passes
    if score > best_score:
        best_score = score
        best_results = cfg_results
        best_cfg_id = cfg['id']

print('=' * 80)
print('MULTI-CONFIGURATION COMPARISON TABLE')
print('=' * 80)
for s in all_eval_summaries:
    print(f"{s['name']:38s} | Leaks: {s['scaffolding_leaks']}/8 | Grounding: {s['pass_rate']}")

with open('smoke_test_results.json', 'w') as f:
    json.dump(best_results, f, indent=2)

with open('smoke_test_comparison.json', 'w') as f:
    json.dump({
        'summaries': all_eval_summaries,
        'best_config': best_cfg_id,
    }, f, indent=2)
print("Saved smoke_test_results.json and smoke_test_comparison.json successfully!")
